<a href="https://colab.research.google.com/github/alexj11324/TSSPIM-Tsunami-and-Storm-Surge-Population-Impact-Model/blob/migrate/notebooks/shelter_demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Storm Surge Shelter Demand Model — Full E2E Pipeline

**American Red Cross — Storm Surge Shelter Demand Estimation**

This notebook runs the complete pipeline from NHC advisory to geography-level impact estimates:

1. **Download** NHC P-Surge raster for the specified storm/advisory
2. **Download** NSI building inventory for affected states
3. **Filter & Map** buildings via DuckDB (bbox clip, dedup, column mapping)
4. **Run FAST** headlessly to produce building-level damage predictions
5. **Aggregate** predictions and join supporting geography/SVI data
6. **Estimate** impacted population and export CSV/XLSX outputs

**How to use:**
1. Update the Excel config workbook in Google Drive or under `data/`
2. Adjust the Google Drive path in the config cell if your shared-drive location differs
3. Run all cells
4. Copy the exported output back into the workbook if needed


In [ ]:
import subprocess, sys, os, io, re, json, time, warnings
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
    'duckdb', 'pyarrow', 'rasterio', 'geopandas',
    'requests', 'openpyxl', 'pygris', 'tqdm', 'utm', 'huggingface_hub'])

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
import utm
from rasterio.warp import transform_bounds
from shapely.geometry import box

In [ ]:
# Optional Google Drive mount for shared ARC workbook access.
try:
    from google.colab import drive
except ImportError:
    GDRIVE_MOUNTED = False
    print('Google Drive mount skipped (not running in Colab).')
else:
    drive.mount('/content/drive', force_remount=True)
    GDRIVE_MOUNTED = True


In [ ]:
REPO_URL = 'https://github.com/alexj11324/TSSPIM-Tsunami-and-Storm-Surge-Population-Impact-Model.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('repo')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )
else,
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)

WORK_DIR = REPO_DIR.resolve() / 'outputs'
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Repo root on sys.path so `from scripts.*` works in all later cells.
repo_root = REPO_DIR.resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Local dev fallback: if running from notebooks/, try cwd or parent for scripts/.
if not (repo_root / 'scripts' / 'nsi_downloader.py').is_file():
    for _cand in (Path.cwd(), Path.cwd().parent):
        if (_cand / 'scripts' / 'nsi_downloader.py').is_file():
            _resolved = str(_cand.resolve())
            if _resolved not in sys.path:
                sys.path.insert(0, _resolved)
            break

print(f'Work dir: {WORK_DIR}')
print(f'Environment ready. Branch={REPO_BRANCH}')


## Config

In [ ]:
# Cell 2: Configuration
# =========================================================
# Reads the Interface sheet of the Excel config file.
# Priority: Google Drive (shared with ARC staff) -> repo fallback.

from scripts.read_excel_config import load_config_from_excel

EXCEL_FILENAME = 'ARC Storm Surge Shelter Demand.xlsx'
GDRIVE_EXCEL_CANDIDATES = [
    Path('/content/drive/MyDrive/Red Cross Capstone Project/Final Report and Deliverables') / EXCEL_FILENAME,
    Path('/content/drive/MyDrive') / EXCEL_FILENAME,
]
REPO_EXCEL = REPO_DIR / 'data' / EXCEL_FILENAME

excel_config_path = None
if GDRIVE_MOUNTED:
    for candidate in GDRIVE_EXCEL_CANDIDATES:
        if candidate.exists():
            excel_config_path = candidate
            print(f'Reading config from Google Drive: {candidate}')
            break

if excel_config_path is None:
    excel_config_path = REPO_EXCEL
    print(f'Reading config from repo: {excel_config_path}')

params = load_config_from_excel(excel_config_path)

if params['geography'].lower() not in ('census tract', 'county'):
    raise ValueError(
        f"geography must be 'census tract' or 'county', got {params['geography']!r}"
    )

if params['flood_load_condition'] not in ('CoastalA', 'CoastalV', 'Riverine'):
    raise ValueError(
        'flood_load_condition must be CoastalA/CoastalV/Riverine, '
        f"got {params['flood_load_condition']}"
    )

for key in ('fast_timeout', 'download_timeout', 'svi_timeout'):
    if params[key] <= 0:
        raise ValueError(f'{key} must be > 0, got {params[key]}')

for key in ('svi_rest_page_size',):
    if not isinstance(params[key], int) or params[key] <= 0:
        raise ValueError(f'{key} must be a positive integer, got {params[key]!r}')

print(
    f"Storm: {params['storm_name']} ({params['storm_id']}), "
    f"Advisory #{params['advisory']}, Year {params['year']}"
)
print(
    f"Geography: {params['geography']}"
)
print(
    f"Residential Building Types Included: {params['BUILDING_TYPES']}"
)
print(
    f"Damage Assessment Categories: {params['DAMAGE_CATEGORIES']}"
)
print('Params validated OK.')


## Stage 1: Download NHC P-Surge Raster

Downloads the storm surge inundation raster directly from NHC for the specified storm and advisory.
Also identifies which US states are within the raster footprint.

In [ ]:
# Cell 3: Download NHC P-Surge Raster
from scripts.import_nhc_by_storm import download_surge_raster

raster_path, affected_states = download_surge_raster(
    params['storm_id'],
    params['storm_name'],
    params['advisory'],
    params['year'],
    output_dir=WORK_DIR,
    timeout=params['download_timeout'],
)

print(f'Raster saved: {raster_path}')
print(f'Affected states: {affected_states}')

with rasterio.open(raster_path) as src:
    data = src.read(1)
    print("Unique codes:", np.unique(data))
    print("Nodata value:", src.nodata)

## Stage 2: Download NSI Building Inventory

Downloads NSI (National Structure Inventory) for each affected state.
The `cbfips` field is critical for census tract GEOID derivation.

**Two download sources available** (set `NSI_SOURCE` below):
- `"usace"` (default) — Stream from USACE API (no token needed, slower for large states)
- `"huggingface"` — Download pre-processed Parquet from [Alexq847182/NSI_Parquet](https://huggingface.co/datasets/Alexq847182/NSI_Parquet) (faster, optional HF token for private/gated datasets)

NSI_SOURCE below is a notebook-only switch; CLI scripts default to the USACE API.


In [ ]:
# Cell 4: Download NSI Data
# ========================
# Set NSI_SOURCE to choose download backend:
#   "usace"       — USACE API (default, no token needed)
#   "huggingface" — HuggingFace dataset Alexq847182/NSI_Parquet
NSI_SOURCE = "huggingface"  # <-- change to "huggingface" to use HF

# Optional: set HF token for private/gated datasets (leave None for public)
# or paste your token: "hf_..."
HF_TOKEN = None #TODO


from scripts.nsi_downloader import NSIDownloader

with rasterio.open(raster_path) as src:
    bounds = src.bounds
    raster_bbox_poly = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    if src.crs and str(src.crs) != "EPSG:4326":
        b = transform_bounds(src.crs, "EPSG:4326", *bounds)
        raster_bbox_poly = box(*b)

downloader = NSIDownloader(WORK_DIR)

if NSI_SOURCE == "huggingface":
    nsi_df = downloader.download_states_hf(affected_states, token=HF_TOKEN)
else:
    nsi_df = downloader.download_states(
        affected_states, raster_bbox_polygon=raster_bbox_poly
    )

print(f"\nTotal NSI buildings ({NSI_SOURCE}): {len(nsi_df):,}")


In [ ]:
# Cell 5: Spatial Filter + FAST Input Preparation

from scripts.duckdb_fast_pipeline import FOUND_TYPE_MAP, FOUND_TYPE_DEFAULT

bbox = raster_bbox_poly.bounds  # (minx, miny, maxx, maxy)
print(f'Raster bbox (WGS84): lon=[{bbox[0]:.4f}, {bbox[2]:.4f}], lat=[{bbox[1]:.4f}, {bbox[3]:.4f}]')

nsi_filtered = nsi_df[
    nsi_df['latitude'].between(bbox[1], bbox[3]) &
    nsi_df['longitude'].between(bbox[0], bbox[2]) &
    nsi_df['latitude'].notna() &
    nsi_df['bid'].notna()
].copy()

nsi_filtered = (
    nsi_filtered.sort_values('val_struct', ascending=False)
    .drop_duplicates(subset='bid', keep='first')
)

# Normalize cbfips to 15-digit strings to preserve leading zeros in downstream joins
nsi_filtered['cbfips'] = (
    nsi_filtered['cbfips']
    .astype('string')
    .str.replace(r"[^0-9]", '', regex=True)
    .str.zfill(15)
)

fast_input = pd.DataFrame({
    'FltyId': nsi_filtered['bid'],
    'Occ': nsi_filtered['occtype'].str.split('-').str[0].str.upper(),
    'Cost': nsi_filtered['val_struct'].fillna(0),
    'Area': nsi_filtered['sqft'].fillna(0),
    'NumStories': nsi_filtered['num_story'].fillna(1),
    'FoundationType': nsi_filtered['found_type'].astype(str).str.upper().str.strip().map(FOUND_TYPE_MAP).fillna(FOUND_TYPE_DEFAULT).astype(int),
    'FirstFloorHt': nsi_filtered['found_ht'].fillna(1.0),
    'ContentCost': nsi_filtered['val_cont'].fillna(0),
    'Latitude': nsi_filtered['latitude'],
    'Longitude': nsi_filtered['longitude'],
})

fast_csv_path = str(WORK_DIR / 'fast_input.csv')
fast_input.to_csv(fast_csv_path, index=False)
print(f'\nFAST input: {len(fast_input):,} buildings -> {fast_csv_path}')
print(f'  Residential: {fast_input["Occ"].str.startswith("RES").sum():,}')
nsi_cbfips_join_path = str(WORK_DIR / 'nsi_cbfips_join.csv')
nsi_filtered[['bid', 'cbfips']].rename(columns={'bid': 'fltyid'}).to_csv(nsi_cbfips_join_path, index=False)
print(f'  cbfips join table -> {nsi_cbfips_join_path}')
fast_input.head()

In [ ]:
# Cell 6: Run FAST Engine

mapping = {
    'UserDefinedFltyId': 'FltyId', 'OCC': 'Occ', 'Cost': 'Cost',
    'Area': 'Area', 'NumStories': 'NumStories',
    'FoundationType': 'FoundationType', 'FirstFloorHt': 'FirstFloorHt',
    'ContentCost': 'ContentCost', 'Latitude': 'Latitude', 'Longitude': 'Longitude',
}

mapping_path = str(WORK_DIR / 'fast_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump(mapping, f)

# --- Pre-flight validation ---
csv_columns = pd.read_csv(fast_csv_path, nrows=0).columns.tolist()
missing = [v for v in mapping.values() if v and v not in csv_columns]
if missing:
    raise ValueError(f'CSV missing mapped columns: {missing}\nCSV has: {csv_columns}')
csv_row_count = len(fast_input)
if csv_row_count <= 0:
    raise ValueError(f'fast_input.csv is empty (0 rows after spatial filtering)')
print(f'Pre-flight OK: {csv_row_count:,} rows, all mapped columns present')

fast_output_dir = str(WORK_DIR / 'fast_output')
os.makedirs(fast_output_dir, exist_ok=True)

fast_script = str(REPO_DIR / 'FAST-main' / 'Python_env' / 'run_fast.py')
cmd = [
    sys.executable, fast_script,
    '--inventory', fast_csv_path,
    '--mapping-json', mapping_path,
    '--flc', params['flood_load_condition'],
    '--rasters', raster_path,
    '--output-dir', fast_output_dir,
    '--project-root', str(REPO_DIR / 'FAST-main'),
]

print('Running FAST engine...')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=params['fast_timeout'])

# --- Parse FAST result (stdout may contain non-JSON print output) ---
fast_payload = None
for line in reversed(result.stdout.strip().splitlines()):
    try:
        fast_payload = json.loads(line)
        break
    except json.JSONDecodeError:
        continue

if result.returncode != 0:
    error_detail = ''
    if fast_payload:
        error_detail = fast_payload.get('error') or fast_payload.get('message', '')
    if not error_detail:
        error_detail = result.stdout[-2000:] if result.stdout else '(empty stdout)'
    print(f'FAST error detail:\n{error_detail}')
    print(f'FAST STDERR (last 2000 chars):\n{result.stderr[-2000:]}')
    raise RuntimeError(f'FAST failed (rc={result.returncode}): {error_detail}')

# --- Check row-level errors on success ---
row_errors = fast_payload.get('row_errors', 0) if fast_payload else 0
if row_errors > 0:
    raise RuntimeError(f'FAST completed but {row_errors} rows had processing errors — results are incomplete')

print('FAST completed successfully.')

# Find predictions output
pred_files = sorted(Path(fast_output_dir).rglob('*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not pred_files:
    raise FileNotFoundError(f'FAST produced no output CSV in {fast_output_dir}')

predictions_path = str(pred_files[0])
print(f'Predictions file saved to: {predictions_path}')

In [ ]:
# Cell 7: Load Predictions + Derive Census GEOID

pred_df = pd.read_csv(predictions_path, dtype={'FltyId': str, 'DebrisID': str})
print(f'Loaded {len(pred_df):,} predictions')

if pred_df.empty:
    raise RuntimeError('Predictions file is empty. FAST may have failed silently.')

pred_df.columns = pred_df.columns.str.lower()

# Dedup: MAX(bldgdmgpct) per FltyId
pred_df = (
    pred_df.sort_values('bldgdmgpct', ascending=False)
    .drop_duplicates(subset='fltyid', keep='first')
)
print(f'After dedup: {len(pred_df):,} unique buildings')

# JOIN with NSI to get cbfips (written in Cell 5 alongside FAST input)
_nsi_join = WORK_DIR / 'nsi_cbfips_join.csv'
if not _nsi_join.is_file():
    raise FileNotFoundError(f'Missing {_nsi_join}. Run Cell 5 first to build FAST input + cbfips join table.')
nsi_cbfips = pd.read_csv(_nsi_join, dtype={'fltyid': str, 'cbfips': str})

pred_df = pred_df.merge(nsi_cbfips, on='fltyid', how='left')
matched = pred_df['cbfips'].notna().sum()
total = len(pred_df)
print(f'Census block FIPS matched: {matched:,} / {total:,} ({matched/max(total,1)*100:.1f}%)')

# Derive tract GEOID (first 11 digits of 15-digit cbfips)
pred_df['cbfips'] = pred_df['cbfips'].astype(str).str.zfill(15)
pred_df['tract_geoid'] = pred_df['cbfips'].str[:11]
pred_df['county_geoid'] = pred_df['cbfips'].str[:5]

# Filter to residential only
res_types = [idx for idx in params['BUILDING_TYPES'].keys() if params['BUILDING_TYPES'][idx] == "Y"]
print(res_types)

pred_df = pred_df[pred_df['occ'].isin(res_types)].copy()
print(f'Residential with tract GEOID: {len(pred_df):,}, unique counties: {pred_df["county_geoid"].nunique():,}, unique tracts: {pred_df["tract_geoid"].nunique():,}')

In [ ]:
# Cell 8: Classify Damage States + Aggregate to Specified Geography

def classify_damage_state(
    depth: pd.Series, categories: dict = None
) -> pd.Series:
    """Classify building damage."""
    if categories is None:
        categories = list(params['DAMAGE_CATEGORIES'].values())
    conditions = [
        depth >= 9,
        depth >= 6,
        depth >= 3,
        depth >= 1,
    ]
    choices = categories
    return np.select(conditions, choices, default='None')

# Classify each building
pred_df['damage_category'] = classify_damage_state(pred_df['depth_grid'])
categories = list(params['DAMAGE_CATEGORIES'].values())

state_dist = pred_df['damage_category'].value_counts()
print('Building damage category distribution:')
for ds in categories:
    n = state_dist.get(ds, 0)
    print(f'  {ds:>10s}: {n:>8,} ({n/len(pred_df)*100:.1f}%)')

# One-hot encode damage states for fast aggregation
for ds in categories:
    pred_df[f'is_{ds}'] = (pred_df['damage_category'] == ds).astype(int)

# One-hot encode storm surge height for fast aggregation
hts = [0, 1, 3, 6, 9]
for ht in hts:
    pred_df[f'ht_{ht}'] = (pred_df['depth_grid'] == ht).astype(int)

# Create indicators for building damage percentages by 10pp bins
for i in range(10):
    pred_df[f'building_damage_{(i+1)*10}pct'] = (pred_df['bldgdmgpct'].between(i*10, (i+1)*10, inclusive = 'right')).astype(int)

# Aggregate to specified geography
if params['geography'].lower() == "census tract":
    geo = 'tract_geoid'
elif params['geography'].lower() == "county":
    geo = 'county_geoid'
pred_df['FIPS'] = pred_df[geo]
geo_agg = pred_df.groupby(['FIPS']).agg(
    Total_Num_Building = ('fltyid', 'count'),

    ## damage states
    Total_Num_Building_cat0 = (f'is_{categories[0]}', 'sum'),
    Total_Num_Building_cat1 = (f'is_{categories[1]}', 'sum'),
    Total_Num_Building_cat2 = (f'is_{categories[2]}', 'sum'),
    Total_Num_Building_cat3 = (f'is_{categories[3]}', 'sum'),

    ## surge heights
    Total_Num_Building_SurgeHeight_ht0 = (f'ht_{hts[0]}', 'sum'),
    Total_Num_Building_SurgeHeight_ht1 = (f'ht_{hts[1]}', 'sum'),
    Total_Num_Building_SurgeHeight_ht2 = (f'ht_{hts[2]}', 'sum'),
    Total_Num_Building_SurgeHeight_ht3 = (f'ht_{hts[3]}', 'sum'),
    Total_Num_Building_SurgeHeight_ht4 = (f'ht_{hts[4]}', 'sum'),

    ## building damage percent
    Building_Damage_Pct_Min  = ('bldgdmgpct', 'min'),
    Building_Damage_Pct_Max  = ('bldgdmgpct', 'max'),
    Building_Damage_Pct_Mean = ('bldgdmgpct', 'mean'),
    Building_Damage_Pct_25th = ('bldgdmgpct', lambda x: x.quantile(0.25)),
    Building_Damage_Pct_50th = ('bldgdmgpct', lambda x: x.quantile(0.50)),
    Building_Damage_Pct_75th = ('bldgdmgpct', lambda x: x.quantile(0.75)),
    Total_Num_Building_Damage_10pct = ('building_damage_10pct', 'sum'),
    Total_Num_Building_Damage_20pct = ('building_damage_20pct', 'sum'),
    Total_Num_Building_Damage_30pct = ('building_damage_30pct', 'sum'),
    Total_Num_Building_Damage_40pct = ('building_damage_40pct', 'sum'),
    Total_Num_Building_Damage_50pct = ('building_damage_50pct', 'sum'),
    Total_Num_Building_Damage_60pct = ('building_damage_60pct', 'sum'),
    Total_Num_Building_Damage_70pct = ('building_damage_70pct', 'sum'),
    Total_Num_Building_Damage_80pct = ('building_damage_80pct', 'sum'),
    Total_Num_Building_Damage_90pct = ('building_damage_90pct', 'sum'),
    Total_Num_Building_Damage_100pct = ('building_damage_100pct', 'sum'),

    ## costs of building losses
    Building_Loss_USD_Min  = ('bldglossusd', 'min'),
    Building_Loss_USD_Max  = ('bldglossusd', 'max'),
    Building_Loss_USD_Mean = ('bldglossusd', 'mean'),
    Building_Loss_USD_25th = ('bldglossusd', lambda x: x.quantile(0.25)),
    Building_Loss_USD_50th = ('bldglossusd', lambda x: x.quantile(0.50)),
    Building_Loss_USD_75th = ('bldglossusd', lambda x: x.quantile(0.75)),
).reset_index()


## rename aggregate columns
geo_agg.rename(columns = {'Total_Num_Building_cat0' : f'Total_Num_Building_{categories[0]}',
                          'Total_Num_Building_cat1' : f'Total_Num_Building_{categories[1]}',
                          'Total_Num_Building_cat2' : f'Total_Num_Building_{categories[2]}',
                          'Total_Num_Building_cat3' : f'Total_Num_Building_{categories[3]}',
                          'Total_Num_Building_SurgeHeight_ht0' : f'Total_Num_Building_SurgeHeight_{hts[0]}',
                          'Total_Num_Building_SurgeHeight_ht1' : f'Total_Num_Building_SurgeHeight_{hts[1]}',
                          'Total_Num_Building_SurgeHeight_ht2' : f'Total_Num_Building_SurgeHeight_{hts[2]}',
                          'Total_Num_Building_SurgeHeight_ht3' : f'Total_Num_Building_SurgeHeight_{hts[3]}',
                          'Total_Num_Building_SurgeHeight_ht4' : f'Total_Num_Building_SurgeHeight_{hts[4]}',
                          }, inplace = True)

orig_cols = geo_agg.columns.tolist()

geo_agg['State_FIPS'] = geo_agg['FIPS'].str[:2]
geo_agg['County_FIPS'] = geo_agg['FIPS'].str[:5]

geo_agg = geo_agg[['State_FIPS', 'County_FIPS'] + orig_cols].copy()
geo_agg = geo_agg.loc[:, ~geo_agg.columns.duplicated()]

print(f'\nAggregated to {len(geo_agg):,} geographies ({params['geography'].lower()}-level)')
geo_agg.head()

In [ ]:
# Cell 9: Load SVI data

CDC_SVI_REST_URL = 'https://onemap.cdc.gov/onemapservices/rest/services/SVI/CDC_ATSDR_Social_Vulnerability_Index_2022_USA/FeatureServer/2/query'

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def _download_svi_via_rest() -> pd.DataFrame:
    rows = []
    offset = 0
    page_size = params['svi_rest_page_size']
    t0 = time.time()
    total_bytes = 0

    with tqdm(desc="Downloading CDC SVI (REST)", unit="page") as pbar:
        while True:
            qparams = {
                'where': '1=1',
                'outFields': 'FIPS,RPL_THEMES',
                'returnGeometry': 'false',
                'resultOffset': offset,
                'resultRecordCount': page_size,
                'f': 'pjson',
            }
            resp = requests.get(CDC_SVI_REST_URL, params=qparams, timeout=params['svi_timeout'])
            resp.raise_for_status()
            total_bytes += len(resp.content)
            payload = resp.json()
            features = payload.get('features', [])
            if not features:
                break

            for feature in features:
                attrs = feature.get('attributes', {})
                rows.append({
                    'FIPS': attrs.get('FIPS'),
                    'RPL_THEMES': attrs.get('RPL_THEMES'),
                })

            offset += len(features)
            pbar.update(1)
            elapsed = max(time.time() - t0, 1e-9)
            mb_s = total_bytes / elapsed / 1024 / 1024
            pbar.set_postfix_str(f"{mb_s:.2f} MB/s")

            if not payload.get('exceededTransferLimit'):
                break

    return pd.DataFrame(rows)


def fetch_svi() -> pd.DataFrame:
    if params['geography'] == 'census tract':
        file = 'SVI_2022_US_CensusTracts.csv'
        CDC_SVI_URL = 'https://svi.cdc.gov/Documents/Data/2022/csv/states/SVI_2022_US.csv'
    else:
        file = 'SVI_2022_US_county.csv'
        CDC_SVI_URL = 'https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv'
    svi_path = WORK_DIR / file
    if not svi_path.exists():
        print('  Downloading CDC SVI 2022 data...')
        try:
            resp = requests.get(CDC_SVI_URL, timeout=params['svi_timeout'], stream=True)
            resp.raise_for_status()
            total = int(resp.headers.get('Content-Length', 0) or 0)
            chunk_size = 1024 * 1024
            t_dl = time.time()
            written = 0

            with open(svi_path, 'wb') as f, tqdm(
                total=total if total > 0 else None,
                desc='  CDC SVI CSV',
                unit='B',
                unit_scale=True,
            ) as pbar:
                for chunk in resp.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    f.write(chunk)
                    written += len(chunk)
                    pbar.update(len(chunk))
                    elapsed = max(time.time() - t_dl, 1e-9)
                    mb_s = written / elapsed / 1024 / 1024
                    pbar.set_postfix_str(f"{mb_s:.2f} MB/s")
        except requests.HTTPError:
            print('  CSV URL unavailable, falling back to CDC ArcGIS REST service...')
            _download_svi_via_rest().to_csv(svi_path, index=False)
        print(f'  Saved to {svi_path}')
    else:
        print('  Loading cached SVI data...')

    svi_vars = ['ST_ABBR', 'COUNTY', 'FIPS', 'E_TOTPOP',
                'EP_POV150', 'EP_HBURD', 'EP_DISABL',
                'EP_MUNIT', 'EP_MOBILE', 'EP_CROWD',
                'EP_NOVEH', 'EP_UNEMP', 'EP_NOHSDP',
                'EP_UNINSUR', 'EP_AGE65', 'EP_AGE17',
                'EP_SNGPNT', 'EP_LIMENG', 'EP_MINRTY',
                'EP_GROUPQ']
    df = pd.read_csv(svi_path, dtype={'FIPS': str}, usecols=svi_vars)
    return df


# Import SVI data
print('\nFetching CDC SVI data...')
svi_df = fetch_svi()
print(f'SVI data: {len(svi_df):,} rows')

geo_agg_svi = geo_agg.merge(svi_df, on='FIPS', how='left')
geo_agg_svi[['ST_ABBR', 'COUNTY']] = geo_agg_svi.groupby(['State_FIPS', 'County_FIPS'])[['ST_ABBR', 'COUNTY']].ffill().bfill()


svi_matched = (~geo_agg_svi['E_TOTPOP'].isna()).sum()
print(f'\nMatched: {svi_matched}/{len(geo_agg_svi)} ({round(100*(svi_matched/len(geo_agg_svi)), 1)}%) with SVI')
print(f'Missing State abbreviation: {(geo_agg_svi['ST_ABBR'].isna()).sum()}')
geo_agg_svi.head()

In [ ]:
# Cell 10: Compute Shelter-Seeking Population

id_cols = ['ST_ABBR', 'COUNTY', 'State_FIPS', 'County_FIPS', 'FIPS']
n_bldg_cols    = [col for col in geo_agg_svi.columns if col.startswith('Total_Num_Building')]
bldg_dmg_cols  = [col for col in geo_agg_svi.columns if col.startswith('Building_Damage')]
bldg_loss_cols = [col for col in geo_agg_svi.columns if col.startswith('Building_Loss')]
svi_cols = ['E_TOTPOP', 'EP_POV150', 'EP_UNEMP', 'EP_HBURD', 'EP_NOHSDP',
            'EP_UNINSUR', 'EP_AGE65', 'EP_AGE17', 'EP_DISABL',
            'EP_SNGPNT', 'EP_LIMENG', 'EP_MINRTY', 'EP_MUNIT',
            'EP_MOBILE', 'EP_CROWD', 'EP_NOVEH', 'EP_GROUPQ']

# Build final output
output_cols = list(dict.fromkeys(id_cols + n_bldg_cols + bldg_dmg_cols + bldg_loss_cols + svi_cols))
final_output = geo_agg_svi[output_cols].copy()
final_output.head(10)

In [ ]:
# Cell 11: Estimates Overview
n_tracts = len(final_output)
total_pop = final_output['E_TOTPOP'].sum()

w = 110
print(f'{"":=<{w}}')
print(f'{"POPULATION IMPACTED MODEL \u2014 OVERVIEW":^{w}}')
print(f'{"":=<{w}}')
print(f'  Storm: {params["storm_name"]} ({params["storm_id"]}), Advisory #{params["advisory"]}')
print(f'  States in Storm Surge Region: {", ".join(affected_states)}')
print(f'{"":_<{w}}')
print(f' Population Impacts')
print(f'  Total Population In Region:  {total_pop:>12,.0f}')
print(f'{"":_<{w}}')

print(f' Building Damage Assessment')
for cat in categories:
    col = f'Total_Num_Building_{cat}'
    total = final_output[col].sum()
    pct = total / max(final_output['Total_Num_Building'].sum(), 1) * 100
    print(f'  {cat:>10s}: {total:>10,.0f} buildings ({pct:.1f}%)')

print(f'{"":=<{w}}')

In [ ]:
# Cell 12: Map Region Impacts

import geopandas as gpd
import matplotlib.pyplot as plt
import pygris

graph_data = final_output.copy()
graph_data['worst_damage_assessment'] = graph_data[f'Total_Num_Building_{categories[0]}'] + graph_data[f'Total_Num_Building_{categories[1]}']

states = graph_data['ST_ABBR'].unique().tolist()

state_bnds = pygris.states(cb=True, resolution="20m")
state_bnds = state_bnds[state_bnds['STUSPS'].isin(states)].copy()
counties = pygris.counties(state=states, cb=True)
if params['geography'] == 'census tract':
    boundaries = pd.DataFrame()
    for st in states:
        tracts = pygris.tracts(state=st)
        boundaries = pd.concat([boundaries, tracts])
else:
    boundaries = counties


fig, ax = plt.subplots(figsize=(12, 12))

merged = boundaries.merge(graph_data, left_on='GEOID', right_on='FIPS')
merged.plot(column='worst_damage_assessment', ax=ax, legend=True, cmap='OrRd',
            linewidth=0.8, edgecolor='0.8').axis('off')

counties.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.5)
state_bnds.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1)
plt.savefig(WORK_DIR / f'{params['storm_name']}_{params['storm_id']}_{params['advisory']}_{params['geography']}.png')

In [ ]:
# Cell 14: Export Results

storm_suffix = f"{params['storm_name']}_{params['storm_id']}_{params['advisory']}_{params['geography']}"

csv_base = Path(str(params['output_csv_name']))
csv_filename = f"{csv_base.stem}_{storm_suffix}{csv_base.suffix or '.csv'}"
csv_path = WORK_DIR / csv_filename
final_output.to_csv(csv_path, index=False)
print(f'CSV exported: {csv_path} ({len(final_output)} rows)')

xlsx_base = Path(str(params['output_xlsx_name']))
xlsx_filename = f"{xlsx_base.stem}_{storm_suffix}{xlsx_base.suffix or '.xlsx'}"
xlsx_path = WORK_DIR / xlsx_filename
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    final_output.to_excel(writer, sheet_name='Output', index=False)
    pd.DataFrame({
        'Parameter': [
            'Storm ID', 'Storm Name', 'Advisory #', 'Year',
            'Total Population in Region',
        ],
        'Value': [
            params['storm_id'], params['storm_name'], params['advisory'], params['year'],
            total_pop,
        ],
    }).to_excel(writer, sheet_name='Parameters', index=False)

print(f'Excel exported: {xlsx_path}')

# JSON for pasting back into Excel Step 7
json_output = final_output.to_json(orient='records', double_precision=6)
print('\n--- JSON output (first 500 chars) ---')
print(json_output[:500])

try:
    from google.colab import files
    files.download(str(csv_path))
    files.download(str(xlsx_path))
except ImportError:
    print(f'\nFiles saved to: {csv_path}, {xlsx_path}')

---

## Model Card

**Model**: Shelter Demand Estimator v1.0

**Data Sources**: NHC P-Surge, NSI (USACE), FEMA FAST, Census ACS 5-year, CDC SVI 2022